Parsing Documents

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Mon" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "W5" / "Mon"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

프로젝트 루트  : d:\hanwha-agent


In [2]:
SAMPLES=SANDBOX/'samples'   # 실습 재료 배치할 폴더

In [3]:
# 파일, 폴더를 통째로 복사/이동하는 표준 라이브러리
import shutil

# 다운받은 경로, 프로젝트 폴더 옆에 seed_docs 압축 해제한 경우
CANDIDATES = [pathlib.Path.home() / 'Downloads' / 'seed_docs', ROOT.parent / 'seed_docs']
SAMPLES.mkdir(parents=True, exist_ok=True) # 샘플 폴더 없으면 생성
found = [p for p in SAMPLES.rglob('*') if p.is_file()] # 파일들 조회

# 이미 파일들이 있으면 몇 개인지 출력
if found:
    print(f'이미 있습니다 : {SAMPLES.relative_to(ROOT)}')
    print(f'파일 {len(found)}개')

# 파일들이 없으면 복사
else:
    SRC = next((c for c in CANDIDATES if c.is_dir()), None)
    if SRC is None:
        print('재료 폴더를 찾지 못했습니다. 아래 두 자리 중 한 곳에 풀어 주세요 —')
        for c in CANDIDATES:
            print('   ', c)
    else:
        shutil.copytree(SRC, SAMPLES, dirs_exist_ok=True)
        print(f'복사 완료 : {SAMPLES.relative_to(ROOT)}')
        print(f"파일 {sum((1 for p in SAMPLES.rglob('*') if p.is_file()))}개")


이미 있습니다 : sandbox\W5\Mon\samples
파일 30개


In [4]:
from collections import Counter # 항목별 개수 세는 표준 라이브러리

ORDER=['.docx','.pdf','.hwpx']

# 파일들만 모아서 확장자(소문자 변환) 기준으로 카운트
counts=Counter(p.suffix.lower() for p in SAMPLES.rglob('*') if p.is_file()) # 전체 조회 반복문
others=sum(n for ext, n in counts.items() if ext not in ORDER) # for~in문
print('counts\n', counts)
print('others\n', others)
print('-----------------------------')
print(f'{sum(counts.values())} files')
print(''+','.join(f'{ext}{counts[ext]}' for ext in ORDER) + f', and then {others} others')

counts
 Counter({'.pdf': 11, '.docx': 10, '.hwpx': 3, '.yaml': 1, '.md': 1, '.txt': 1, '.pptx': 1, '.html': 1, '.xlsx': 1})
others
 6
-----------------------------
30 files
.docx10,.pdf11,.hwpx3, and then 6 others


In [5]:
from pypdf import PdfReader 
reader = PdfReader(SAMPLES / 'DOC-HR-014_국내출장_여비_규정_v2.0.pdf') 
reader.pages                     # 페이지들 리스트 
reader.pages[0].extract_text()   # 해당 페이지의 글자를 하나의 문자열로 추출 

'일반\n 국내출장 여비 규정\n v2.0\n 문서번호 DOC-HR-014\n 시행일 2025-07-01\n 소관부서 인사총무팀\n 문서책임자 인사총무팀장\n 이 문서는 사내 임직원에 한하여 열람할 수 있다.\n'

PDF

In [6]:
from pypdf import PdfReader

# Path.glob('패턴'): 폴더 안에서 이름이 패턴에 맞는 파일을 찾아 하나씩 내놓는 객체
# next(...): 조회된 파일들 중 첫번째 하나만 꺼내기
PDF=next(SAMPLES.glob('DOC-HR-014_*_v2.0.pdf'))

reader=PdfReader(str(PDF))
print('쪽수:',len(reader.pages))

쪽수: 16


In [7]:
first=reader.pages[0].extract_text()

print(repr(first[:60])) # repr(): 화면에 안보이는 것까지 그대로 보여줌. \n, 앞뒤공백 등
print(first[:60])

'일반\n 국내출장 여비 규정\n v2.0\n 문서번호 DOC-HR-014\n 시행일 2025-07-01\n 소관부서 '
일반
 국내출장 여비 규정
 v2.0
 문서번호 DOC-HR-014
 시행일 2025-07-01
 소관부서 


페이지 단위로 읽기

In [8]:
total=0

for page_no, page in enumerate(reader.pages, 1):
    text=(page.extract_text() or '').strip()
    total+=len(text)
    print(f'p.{page_no} {len(text)}자')

print('-'*15)
print(f'{len(reader.pages)} 페이지, 합계 약 {round(total,2):,}자')

p.1 108자
p.2 231자
p.3 443자
p.4 272자
p.5 1070자
p.6 903자
p.7 1005자
p.8 1042자
p.9 961자
p.10 222자
p.11 281자
p.12 282자
p.13 310자
p.14 244자
p.15 211자
p.16 216자
---------------
16 페이지, 합계 약 7,801자


In [9]:
SCAN=SAMPLES/'_formats'/'sample_scanned.pdf'

scan_reader=PdfReader(str(SCAN))
print('쪽수:', len(scan_reader.pages))

total=0

for page_no, page in enumerate(scan_reader.pages, 1):
    text=(page.extract_text() or '').strip()
    total+=len(text)
    print(f'p.{page_no} {len(text)}자')

print('-'*15)
print(f'{len(scan_reader.pages)} 페이지, 합계 약 {round(total,2):,}자')

쪽수: 1
p.1 0자
---------------
1 페이지, 합계 약 0자


In [10]:
from app.core.logging import get_logger

log = get_logger('app.rag.local_parsers')

blocks = []
for page_no, page in enumerate(scan_reader.pages, 1):
    text = (page.extract_text() or '').strip()
    if not text:
        log.info('텍스트 없음 (스캔본으로 보임): %s p.%d', SCAN.name, page_no)
        continue
    blocks.append(text)
print(f'블록 {len(blocks)}개')

14:50:25 INFO     app.rag.local_parsers: 텍스트 없음 (스캔본으로 보임): sample_scanned.pdf p.1
블록 0개


In [11]:
from app.rag.local_parsers import parse_pdf

doc = parse_pdf(PDF)

print(f'블록 {len(doc.blocks)} · page_count {doc.page_count} · table_count {doc.table_count}')
sixth = doc.blocks[5]
print(f'여섯 번째 블록 : kind={sixth.kind}  locator={sixth.locator}')
print()
print('※ ImportError 가 나면 Kernel ▸ Restart Kernel 뒤 맨 위부터 다시 실행하세요.')

블록 16 · page_count 16 · table_count 0
여섯 번째 블록 : kind=조항  locator=p.6

※ ImportError 가 나면 Kernel ▸ Restart Kernel 뒤 맨 위부터 다시 실행하세요.


In [12]:
# 스캔 PDF도 테스트-> 로그 나오는지 확인하기
doc=parse_pdf(SCAN)
print(f'블럭:{len(doc.blocks)} / page_count:{doc.page_count} / table_count:{doc.table_count}')

14:50:25 INFO     app.rag.local_parsers: 텍스트 없음 (스캔본으로 추정됨): sample_scanned.pdf p.1
블럭:0 / page_count:1 / table_count:0


In [13]:
import zipfile

DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
print('첫 4바이트 :', DOCX.read_bytes()[:4], '  ← ZIP 표식')

with zipfile.ZipFile(DOCX) as zf:
    names = zf.namelist()
print(f'엔트리 {len(names)}개')

for name in names[:6]:
    mark = '  ← 본문이 여기 있습니다' if name == 'word/document.xml' else ''
    print(f'  {name}{mark}')


첫 4바이트 : b'PK\x03\x04'   ← ZIP 표식
엔트리 17개
  [Content_Types].xml
  _rels/.rels
  docProps/core.xml
  docProps/app.xml
  word/document.xml  ← 본문이 여기 있습니다
  word/_rels/document.xml.rels


In [14]:
from docx import Document   # python-docx 라이브러리 

doc = Document(str(DOCX))     # 문서 읽어오기
# doc.paragraphs                   # 문단 목록 
# doc.tables                       # 표 목록
print(f'문단: {len(doc.paragraphs)}개')
print(f'표: {len(doc.tables)}개')
print('첫 문단:', doc.paragraphs[0].text)

문단: 246개
표: 7개
첫 문단: 일반


빈 문단은 빼고 진짜 글자가 있는 문단만 남기기

In [15]:
texts=[p.text.strip() for p in doc.paragraphs if p.text.strip()]
print(f'글자가 있는 문단 개수:{len(texts)}개')
print(f'빈 문단:{len(doc.paragraphs)-len(texts)}개') # 문서에서 엔터만 친 공백들
print(texts[:3])

글자가 있는 문단 개수:226개
빈 문단:20개
['일반', '국내출장 여비 규정', 'v2.0']


In [16]:
bad = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

table_pos = []

for table in doc.tables:
    cells = [cell.text.strip() for row in table.rows for cell in row.cells]
    bad.append(' '.join((c for c in cells if c)))
    table_pos.append(len(bad))
    
print(f'총 블록 {len(bad)}개')
print(f'표가 들어간 자리 : {table_pos[0]} ~ {table_pos[-1]} 번째  (전부 맨 끝)')


총 블록 233개
표가 들어간 자리 : 227 ~ 233 번째  (전부 맨 끝)


In [17]:
only_paragraphs = sum((len(t) for t in texts))

table_chars = sum((len(cell.text.strip()) for table in doc.tables for row in table.rows for cell in row.cells))

with_tables = only_paragraphs + table_chars

print(f'문단만 읽었을 때 : {only_paragraphs:,}자')
print(f'표까지 읽었을 때 : {with_tables:,}자')
print('─' * 29)
print(f'표 안 글자 {table_chars:,}자  =  전체의 {table_chars / with_tables:.1%}')
print()
print(f'우리 코드에서 첫 표의 자리 : {table_pos[0]} 번째')


문단만 읽었을 때 : 6,230자


표까지 읽었을 때 : 7,358자
─────────────────────────────
표 안 글자 1,128자  =  전체의 15.3%

우리 코드에서 첫 표의 자리 : 227 번째


In [18]:
from collections import Counter
body = doc.element.body # 본문 xml 요소
tags = []
for child in body.iterchildren(): # 바로 아래 자식들을 원문에 놓인 순서 그대로 (출력?)
    tag = child.tag.split('}')[-1] # child.tag '{긴 주소}p'
    # print('tag:',child.tag)
    if tag not in ('p', 'tbl'):
        continue
    if tag == 'p' and (not ''.join(child.itertext()).strip()):
        continue
    tags.append(tag)
kind = Counter(tags)
print(f'본문 자식 {len(tags)}개')
print(f"종류 : p {kind['p']} · tbl {kind['tbl']}")
print('tbl 위치 :', [i for i, t in enumerate(tags, 1) if t == 'tbl'])


본문 자식 233개
종류 : p 226 · tbl 7
tbl 위치 : [10, 218, 221, 224, 227, 229, 232]


In [19]:
from docx.table import Table
from docx.text.paragraph import Paragraph

blocks=[]
tables=0

for child in body.iterchildren():
    tag=child.tag.split('}')[-1]
    if tag=='p':
        text=Paragraph(child,doc).text.strip()
        if text:
            blocks.append(text)
    elif tag=='tbl':
        table=Table(child,doc)
        tables+=1
        blocks.append(f'[표{tables}]') # 일단 표 번호만 추가 (자리만 잡아두기)

print(f'블록 {len(blocks)}개')
print('10번째 항목:',blocks[9])
print('9번째 항목:',blocks[8])

블록 233개
10번째 항목: [표1]
9번째 항목: 제·개정 이력


In [20]:
path = ROOT / 'backend' / 'app' / 'rag' / 'local_parsers.py'
text = path.read_text(encoding='utf-8')
print(path.name, '·', len(text.splitlines()), '줄')
for name in ['parse_pdf', 'parse_docx']:
    print(f'  def {name:<11}:', text.count(f'def {name}('), '개')

local_parsers.py · 50 줄
  def parse_pdf  : 1 개
  def parse_docx : 1 개


In [21]:
try:
    from app.rag.local_parsers import parse_docx
except ImportError as err:
    print('import 실패 :', err)
    print('→ Kernel ▸ Restart 로 커널을 재시작한 뒤 맨 위 두 셀부터 다시 실행하세요.')
    raise
    
doc = parse_docx(DOCX)
print(f'블록 {len(doc.blocks)} · page_count {doc.page_count} · table_count {doc.table_count}')
print('표 위치(0부터) :', [i for i, b in enumerate(doc.blocks) if b.kind == '표'])
print("(이전의 '10번째' 와 1 차이가 나는 것은 0 부터 세기 때문입니다)")

블록 233 · page_count 1 · table_count 7
표 위치(0부터) : [9, 217, 220, 223, 226, 228, 231]
(이전의 '10번째' 와 1 차이가 나는 것은 0 부터 세기 때문입니다)


In [22]:
"""표의 머리글과 나머지 행들을 마크다운 표 한 덩어리로 바꾼다."""
def _table_to_markdown(headers: list[str], rows: list[list[str]]) -> str: 
    head = '| ' + ' | '.join(headers) + ' |'
    sep = '|' + '---|' * len(headers)
    body = ['| ' + ' | '.join((str(c) for c in r)) + ' |' for r in rows]
    return '\n'.join([head, sep, *body])
    
print(_table_to_markdown(['버전', '구분', '시행일'], [['v2.0', '개정', '2025-07-01']]))

| 버전 | 구분 | 시행일 |
|---|---|---|
| v2.0 | 개정 | 2025-07-01 |


In [23]:
rows = [[cell.text.strip() for cell in row.cells] for row in table.rows]
print(_table_to_markdown(rows[0], rows[1:]))
print()
print('첫 행을 머리로 씁니다 - 머리 행이 없는 표도 마찬가지입니다.')

| 항 목 | 기 재 란 |
|---|---|
| 성명 / 사번 |  |
| 소속 / 직급 |  |
| 출장 기간 |  |
| 출장지 |  |
| 일비 (일수 × 단가) |  |
| 식비 (일수 × 단가) |  |
| 숙박비 (박수 × 실지출) |  |
| 교통비 (구간별 실지출) |  |
| 주차료·통행료 |  |
| 지급 합계 |  |
| 가지급액 |  |
| 추가 지급 / 반환액 |  |
| 법인카드 결제분 |  |
| 첨부 서류 |  |
| 정산일 / 정산자 |  |
| 팀장 확인 |  |
| 인사총무팀 확인 |  |

첫 행을 머리로 씁니다 - 머리 행이 없는 표도 마찬가지입니다.


In [24]:
import importlib
import app.rag.local_parsers as lp
importlib.reload(lp)         # 파일 다시 읽어오기     
parse_docx = lp.parse_docx

SyntaxError: invalid syntax (local_parsers.py, line 24)

In [25]:
from app.rag.local_parsers import parse_docx

doc = parse_docx(DOCX)
print(f"블록 {len(doc.blocks)} / table_count {doc.table_count}")
print()
print("첫 표 블럭의 내용 : ")
first = next(b for b in doc.blocks if b.kind == "표") 
# print(first.text)
# print(first.text.splitlines()) # 하나의 문자열을 줄단위로 분리해 리스트로 받기 
print("\n".join(first.text.splitlines()))

블록 233 / table_count 7

첫 표 블럭의 내용 : 



In [26]:
from pypdf import PdfReader

PDF = next(SAMPLES.glob('DOC-HR-014_*_v2.0.pdf'))

reader = PdfReader(str(PDF))
lines = (reader.pages[1].extract_text() or '').splitlines()

for n, line in enumerate(lines[:25], 1):
    print(f'{n:2d} | {line}')
    
print('...')
total = sum((len((p.extract_text() or '').splitlines()) for p in reader.pages))

print(f'(문서 전체 {total}줄)')

 1 | 제·개정 이력
 2 | 버전
 3 | 구분
 4 | 시행일
 5 | 주요 내용
 6 | 작성
 7 | 검토
 8 | 승인
 9 | v2.0
10 | 개정
11 | 2025-07-01
12 | 일비
13 | 인상(25,000→30,000)
14 | · 광역시 숙박비 상한
15 | 신설 · 별표1 전면 개정
16 | 김지원
17 | 박성호
18 | 대표이사
19 | v1.1
20 | 개정
21 | 2024-09-01
22 | 교통비 정산 방식을 실비
23 | 정산으로 변경
24 | 김지원
25 | 박성호
...
(문서 전체 434줄)


In [27]:
import pdfplumber

with pdfplumber.open(str(PDF)) as pdf:
    tables = pdf.pages[1].extract_tables()
    
t = tables[0]

print(f'이 쪽의 표 : {len(tables)}개')
print(f'{len(t)}행 {len(t[0])}열')
print(t[0])
print(t[1])

이 쪽의 표 : 1개
4행 7열
['버전', '구분', '시행일', '주요 내용', '작성', '검토', '승인']
['v2.0', '개정', '2025-07-01', '일비\n인상(25,000→30,000)\n· 광역시 숙박비 상한\n신설 · 별표1 전면 개정', '김지원', '박성호', '대표이사']


In [28]:
path = ROOT / "backend" / "tests" / "test_local_parsers.py"
text = path.read_text(encoding="utf-8")
print(path.name, "·", len(text.splitlines()), "줄 · 테스트 함수", text.count("\ndef test_"), "개")

test_local_parsers.py · 45 줄 · 테스트 함수 3 개
